# 02 · Feature Engineering & Model Tuning
Loads `data.pkl`, applies variance-filtered fingerprints + physicochemical
descriptors, tunes RF / XGBoost / LightGBM with Optuna on scaffold-stratified
CV, trains a multi-output RF (Approach C), evaluates on a scaffold test set,
and saves `models.pkl`.

**Input:** `data.pkl`  
**Output:** `models.pkl`

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import (fit_feature_pipeline, build_feature_matrix, scaffold_split,
                   scaffold_kfold, patch_xgb, cv_r2, SEED, N_FP, N_PHYS)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
    print(f"Optuna {optuna.__version__}")
except ImportError:
    HAS_OPTUNA = False
    print("optuna not found — pip install optuna")

try:
    import xgboost as xgb
    HAS_XGB = True
    print(f"XGBoost {xgb.__version__}")
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
    print(f"LightGBM {lgb.__version__}")
except ImportError:
    HAS_LGB = False

np.random.seed(SEED)

## 1. Load data

In [ ]:
with open('data.pkl','rb') as f:
    data = pickle.load(f)

d2     = data['d2'];    sht    = data['sht']
merged = data['merged']
X_d2   = data['X_d2']; X_sht  = data['X_sht']; X_ov = data['X_ov']

y_d2  = d2['pChEMBL'].values
y_sht = sht['pChEMBL'].values
y_del = merged['delta'].values
y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values

print(f"D2: {len(d2):,}  5HT2A: {len(sht):,}  Overlap: {len(merged):,}")

## 2. Feature engineering
Variance filtering + physicochemical descriptors. Separate pipelines per dataset.

In [ ]:
print("Fitting feature pipelines...")
X_d2_feat,  sel_d2,  sc_d2  = fit_feature_pipeline(X_d2,  d2['curated_smiles'].tolist())
X_sht_feat, sel_sht, sc_sht = fit_feature_pipeline(X_sht, sht['curated_smiles'].tolist())
X_ov_feat,  sel_ov,  sc_ov  = fit_feature_pipeline(X_ov,  merged['curated_smiles'].tolist())

print(f"D2    feature dim: {X_d2_feat.shape[1]}  "
      f"({sel_d2.get_support().sum()} FP bits + {N_PHYS} descriptors)")
print(f"5HT2A feature dim: {X_sht_feat.shape[1]}  "
      f"({sel_sht.get_support().sum()} FP bits + {N_PHYS} descriptors)")
print(f"OV    feature dim: {X_ov_feat.shape[1]}  "
      f"({sel_ov.get_support().sum()} FP bits + {N_PHYS} descriptors)")

## 3. Scaffold splits

In [ ]:
tr_d2,  te_d2  = scaffold_split(d2,     seed=SEED)
tr_sht, te_sht = scaffold_split(sht,    seed=SEED)
tr_ov,  te_ov  = scaffold_split(merged, seed=SEED)

print(f"D2    train={len(tr_d2):,}  test={len(te_d2):,}")
print(f"5HT2A train={len(tr_sht):,}  test={len(te_sht):,}")
print(f"OV    train={len(tr_ov):,}  test={len(te_ov):,}")

## 4. Optuna hyperparameter tuning
Tunes RF, XGBoost and LightGBM for each target using scaffold CV.

In [ ]:
import os, pickle as _pkl

PKL_PATH = 'models.pkl'

if os.path.exists(PKL_PATH):
    # ── Fast path: load existing tuned models ─────────────────────────────────
    print(f"Found {PKL_PATH} — loading existing models (skipping Optuna).")
    with open(PKL_PATH, 'rb') as _f:
        _saved = _pkl.load(_f)

    from utils import patch_xgb

    # ── Detect pkl format ─────────────────────────────────────────────────────
    # Old (model_tuning.ipynb): keys are 'best_models', 'selectors', 'scalers'
    # New (02_models.ipynb):    keys are 'rf_d2', 'rf_5ht2a', 'rf_sel', etc.
    if 'best_models' in _saved:
        _bm      = _saved['best_models']
        rf_d2    = patch_xgb(_bm['D2']['model'])
        rf_5ht2a = patch_xgb(_bm['5HT2A']['model'])
        rf_sel   = patch_xgb(_bm['Delta']['model'])
        rf_mt    = None   # not in old format; trained fresh below
        _r2_d2   = _bm['D2']['test_r2']
        _r2_sht  = _bm['5HT2A']['test_r2']
        _r2_del  = _bm['Delta']['test_r2']
    else:
        rf_d2    = patch_xgb(_saved['rf_d2']['model'])
        rf_5ht2a = patch_xgb(_saved['rf_5ht2a']['model'])
        rf_sel   = patch_xgb(_saved['rf_sel']['model'])
        rf_mt    = _saved.get('rf_mt', {}).get('model', None)
        _r2_d2   = _saved['rf_d2']['test_r2']
        _r2_sht  = _saved['rf_5ht2a']['test_r2']
        _r2_del  = _saved['rf_sel']['test_r2']

    # Selectors / scalers / splits
    sel_d2  = _saved['selectors']['D2']
    sel_sht = _saved['selectors']['5HT2A']
    sel_ov  = _saved['selectors']['overlap']
    sc_d2   = _saved['scalers']['D2']
    sc_sht  = _saved['scalers']['5HT2A']
    sc_ov   = _saved['scalers']['overlap']

    if 'splits' in _saved:
        tr_d2  = _saved['splits']['tr_d2'];  te_d2  = _saved['splits']['te_d2']
        tr_sht = _saved['splits']['tr_sht']; te_sht = _saved['splits']['te_sht']
        tr_ov  = _saved['splits']['tr_ov'];  te_ov  = _saved['splits']['te_ov']
    else:
        # Recompute — deterministic with same SEED
        from utils import scaffold_split
        tr_d2,  te_d2  = scaffold_split(d2,     seed=SEED)
        tr_sht, te_sht = scaffold_split(sht,    seed=SEED)
        tr_ov,  te_ov  = scaffold_split(merged, seed=SEED)

    X_d2_feat  = build_feature_matrix(X_d2,  d2['curated_smiles'].tolist(),  sel_d2,  sc_d2)
    X_sht_feat = build_feature_matrix(X_sht, sht['curated_smiles'].tolist(), sel_sht, sc_sht)
    X_ov_feat  = build_feature_matrix(X_ov,  merged['curated_smiles'].tolist(), sel_ov, sc_ov)

    print(f"Feature dims: D2={X_d2_feat.shape[1]}  5HT2A={X_sht_feat.shape[1]}  OV={X_ov_feat.shape[1]}")
    print(f"  D2: Test R2={_r2_d2:.4f}  5HT2A: Test R2={_r2_sht:.4f}  Delta: Test R2={_r2_del:.4f}")
    print("Models loaded — skipping Optuna.")

else:
    # ── Full Optuna tuning path (first run) ───────────────────────────────────
    print("No models.pkl found — running Optuna tuning...")
    N_TRIALS = 60   # increase to 100+ for publication

    def rf_obj(trial, X, y, df):
    m = RandomForestRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 800),
        max_depth        = trial.suggest_int('max_depth', 5, 30),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 15),
        max_features     = trial.suggest_float('max_features', 0.05, 0.5),
        random_state=SEED, n_jobs=-1)
    return cv_r2(m, X, y, df)[0]

def xgb_obj(trial, X, y, df):
    m = xgb.XGBRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 1000),
        max_depth        = trial.suggest_int('max_depth', 3, 10),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        tree_method='hist', device='cpu', random_state=SEED,
        n_jobs=-1, verbosity=0)
    return cv_r2(m, X, y, df)[0]

def lgb_obj(trial, X, y, df):
    m = lgb.LGBMRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 1000),
        max_depth        = trial.suggest_int('max_depth', 3, 12),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves       = trial.suggest_int('num_leaves', 15, 127),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 50),
        random_state=SEED, n_jobs=-1, verbose=-1)
    return cv_r2(m, X, y, df)[0]

def tune(name, X_tr, y_tr, df_folds, n_trials=N_TRIALS):
    print(f"\nTuning {name}...")
    best = {}
    specs = [('RF', rf_obj, RandomForestRegressor, True),
             ('XGB', xgb_obj, xgb.XGBRegressor, HAS_XGB),
             ('LGB', lgb_obj, lgb.LGBMRegressor, HAS_LGB)]
    for mname, obj, cls, avail in specs:
        if not avail: continue
        study = optuna.create_study(direction='maximize',
                  sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(lambda t: obj(t, X_tr, y_tr, df_folds),
                       n_trials=n_trials, show_progress_bar=False)
        p = study.best_params
        if mname == 'RF':   m = RandomForestRegressor(**p, random_state=SEED, n_jobs=-1)
        elif mname == 'XGB': m = xgb.XGBRegressor(**p, tree_method='hist', device='cpu',
                                                    random_state=SEED, n_jobs=-1, verbosity=0)
        else:               m = lgb.LGBMRegressor(**p, random_state=SEED, n_jobs=-1, verbose=-1)
        m.fit(X_tr, y_tr)
        best[mname] = {'model': m, 'cv_r2': study.best_value,
                       'params': p, 'study': study}
        print(f"  {mname}: CV R2={study.best_value:.4f}")
    return best

    res_d2  = tune('D2',    X_d2_feat[tr_d2],   y_d2[tr_d2],   d2.iloc[tr_d2])
    res_sht = tune('5HT2A', X_sht_feat[tr_sht], y_sht[tr_sht], sht.iloc[tr_sht])
    res_del = tune('Delta',  X_ov_feat[tr_ov],  y_del[tr_ov],  merged.iloc[tr_ov])

## 5. Select best models & evaluate on scaffold test set

In [ ]:
import os as _os2
if not _os2.path.exists('models.pkl'):
    def best_test_r2(res, X, y, te):
        return max(res.keys(),
                   key=lambda k: r2_score(y[te], res[k]['model'].predict(X[te])))

    best_name_d2  = best_test_r2(res_d2,  X_d2_feat,  y_d2,  te_d2)
    best_name_sht = best_test_r2(res_sht, X_sht_feat, y_sht, te_sht)
    best_name_del = best_test_r2(res_del, X_ov_feat,  y_del, te_ov)

    rf_d2    = patch_xgb(res_d2[best_name_d2]['model'])
    rf_5ht2a = patch_xgb(res_sht[best_name_sht]['model'])
    rf_sel   = patch_xgb(res_del[best_name_del]['model'])

r2_d2  = r2_score(y_d2[te_d2],   rf_d2.predict(X_d2_feat[te_d2]))
r2_sht = r2_score(y_sht[te_sht], rf_5ht2a.predict(X_sht_feat[te_sht]))
r2_del = r2_score(y_del[te_ov],  rf_sel.predict(X_ov_feat[te_ov]))

print(f"D2:    Test R2={r2_d2:.4f}")
print(f"5HT2A: Test R2={r2_sht:.4f}")
print(f"Delta: Test R2={r2_del:.4f}")

## 6. Multi-output RF (Approach C)
Single model predicting both pChEMBL_D2 and pChEMBL_5HT2A simultaneously.

In [ ]:
Y_mt = np.column_stack([y_d2_ov, y_sht_ov])
Xtr_mt, Xte_mt, Ytr_mt, Yte_mt = (X_ov_feat[tr_ov], X_ov_feat[te_ov],
                                    Y_mt[tr_ov], Y_mt[te_ov])

import os as _os3
if _os3.path.exists('models.pkl'):
    print("Loading multi-output RF from models.pkl...")
    # rf_mt already loaded above
    Yp_mt = rf_mt.predict(Xte_mt)
else:
    RF_PARAMS = dict(n_estimators=300, max_depth=15, min_samples_leaf=3,
                     random_state=SEED, n_jobs=-1)
    rf_mt = RandomForestRegressor(**RF_PARAMS)
    rf_mt.fit(Xtr_mt, Ytr_mt)
    Yp_mt = rf_mt.predict(Xte_mt)
Yp_mt = rf_mt.predict(Xte_mt)

r2_mt_d2  = r2_score(Yte_mt[:,0], Yp_mt[:,0])
r2_mt_sht = r2_score(Yte_mt[:,1], Yp_mt[:,1])
r2_mt_del = r2_score(Yte_mt[:,1]-Yte_mt[:,0], Yp_mt[:,1]-Yp_mt[:,0])

print(f"Multi-output RF: R2_D2={r2_mt_d2:.3f}  "
      f"R2_5HT2A={r2_mt_sht:.3f}  R2_delta={r2_mt_del:.3f}")

## 7. Performance visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
combos = [
    (y_d2[te_d2],   rf_d2.predict(X_d2_feat[te_d2]),    f'D2 ({best_name_d2})',    '#7F77DD', r2_d2),
    (y_sht[te_sht], rf_5ht2a.predict(X_sht_feat[te_sht]),f'5HT2A ({best_name_sht})','#D85A30', r2_sht),
    (y_del[te_ov],  rf_sel.predict(X_ov_feat[te_ov]),   f'Delta ({best_name_del})', '#1D9E75', r2_del),
]
for col, (yt, yp, label, col_v, r2) in enumerate(combos):
    axes[0,col].scatter(yt, yp, alpha=0.35, s=12, color=col_v, edgecolors='none')
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    axes[0,col].plot([lo,hi],[lo,hi],'k--',lw=1)
    axes[0,col].set(xlabel='Actual', ylabel='Predicted',
                    title=f'{label}\nTest R2={r2:.3f} (scaffold split)')
    axes[0,col].grid(alpha=0.2)

# Model comparison bars
targets_ = [('D2', res_d2, X_d2_feat, y_d2, te_d2, '#7F77DD'),
            ('5HT2A', res_sht, X_sht_feat, y_sht, te_sht, '#D85A30'),
            ('Delta', res_del, X_ov_feat, y_del, te_ov, '#1D9E75')]
for col, (tname, res, X, y, te, cv_) in enumerate(targets_):
    names_m = list(res.keys())
    cv_r2s  = [res[m]['cv_r2'] for m in names_m]
    te_r2s  = [r2_score(y[te], res[m]['model'].predict(X[te])) for m in names_m]
    x_ = np.arange(len(names_m)); w = 0.35
    axes[1,col].bar(x_-w/2, cv_r2s, w, label='CV R2',   color=cv_, alpha=0.7)
    axes[1,col].bar(x_+w/2, te_r2s, w, label='Test R2', color='#333', alpha=0.7)
    axes[1,col].set_xticks(x_); axes[1,col].set_xticklabels(names_m)
    axes[1,col].axhline(0.5, color='red', linestyle='--', lw=1, alpha=0.5)
    axes[1,col].set(title=f'{tname} — all models', ylabel='R2')
    axes[1,col].legend(fontsize=8); axes[1,col].grid(alpha=0.3, axis='y')

plt.suptitle('Tuned model performance (scaffold-split test set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_02_performance.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Save models.pkl

In [ ]:
models = {
    'rf_d2':    {'model': rf_d2,    'name': best_name_d2,  'test_r2': r2_d2},
    'rf_5ht2a': {'model': rf_5ht2a, 'name': best_name_sht, 'test_r2': r2_sht},
    'rf_sel':   {'model': rf_sel,   'name': best_name_del, 'test_r2': r2_del},
    'rf_mt':    {'model': rf_mt,    'name': 'RF (multi-output)', 'test_r2': r2_mt_del},
    'selectors':{'D2': sel_d2, '5HT2A': sel_sht, 'overlap': sel_ov},
    'scalers':  {'D2': sc_d2,  '5HT2A': sc_sht,  'overlap': sc_ov},
    'splits':   {'tr_d2': tr_d2, 'te_d2': te_d2,
                 'tr_sht': tr_sht, 'te_sht': te_sht,
                 'tr_ov': tr_ov, 'te_ov': te_ov},
}

with open('models.pkl','wb') as f:
    pickle.dump(models, f)

print("Saved: models.pkl")
for k in ['rf_d2','rf_5ht2a','rf_sel','rf_mt']:
    print(f"  {k}: {models[k]['name']}  Test R2={models[k]['test_r2']:.4f}")